# 11 – Testing NVIDIA's Kumo Tabular

**Objective:** check whether a new pretrained model for tables predicts tennis matches better than our logistic regression.

**Key results**
- **Kumo doesn't pass the test:** on 6,460 matches from 2019 to 2021, its log loss is 0.5951 against our 0.5942. The difference is −0.00083 (95% interval: −0.00342 to +0.00186), so this is effectively a tie.
- **It came very close with far less data:** it saw only 4,000 past matches, against about 35,000 to 39,000 for our model, and needed no training on our data.
- **The two models mostly agree:** they pick the same favourite in 92.9% of matches, so a more powerful model doesn't change many predictions.
- **We keep our model:** it's simpler and easier to explain, and Kumo gives no measured gain.

**Data:** the same features as our model, for main-tour matches up to 2021. The 2022–2026 years aren't used, because they've already been examined. Nothing here changes the final model.

**How to reproduce it:** this notebook reads saved predictions, because Kumo needs its own environment and took hours on a laptop. `src/kumo_test.py` creates them.

In [1]:
import sqlite3
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, "../src")
from tennis_domain_features import DOMAIN_GROUPS
from tennis_features import FEATURE_GROUPS
from tennis_models import INTERACTIONS, add_interactions, logistic_model, score, walk_forward

saved = pd.read_csv("../data/processed/kumo_predictions.csv")
print(f"{len(saved):,} matches, {saved['year'].min()} to {saved['year'].max()}")
saved.head(3)

6,460 matches, 2019 to 2021


,tourney_id,match_num,year,ours,kumo
0,2019-0301,271,2019,0.543879,0.514169
1,2019-0301,272,2019,0.570564,0.627346
2,2019-0301,273,2019,0.715371,0.679348


## 1. What Kumo Tabular is

Kumo Tabular is an open model that NVIDIA released in September 2026. Most models learn from one dataset. Kumo was instead trained beforehand on millions of artificial tables, to learn how to predict on tables in general.

To use it, you give it rows with known results (the **context**) and rows to predict (the **query**). It reads both and predicts directly, with no training on your data. NVIDIA reports first place on several public benchmarks, although no independent head-to-head comparison was found at the time of this test.

## 2. The rule, fixed before opening 2019–2021

| Choice | Decision |
|---|---|
| **Matches** | Main tour, 2019 to 2021 |
| **Columns** | The same 60 for both models |
| **Our model** | Logistic regression, retrained each year on all matches from 2005 |
| **Kumo** | Small size, 8 estimators, the 4,000 latest matches before each year, no calibration |
| **Both models** | Each match is predicted from both points of view and averaged |
| **Pass rule** | The lower end of the 95% bootstrap interval of the log loss gain must be above zero |
| **If it fails** | Keep our model, and try no other Kumo setting on these years as a test |

**Choosing the setting.** Three options were compared on 2018 only, with 1 estimator to save time. The test uses 8, as in NVIDIA's example, to give Kumo its best chance.

| 2018 | Accuracy | Log loss |
|---|---|---|
| Our model | 66.72% | 0.5967 |
| **Kumo small, 4,000 matches (chosen)** | 66.79% | 0.6033 |
| Kumo large, 4,000 matches | 66.53% | 0.6042 |
| Kumo large, 2,000 matches | 66.90% | 0.6158 |

**Why only 4,000 matches?** At 5,000 matches, this laptop's GPU slowed sharply as memory ran short. Longer runs on the GPU also filled the memory, so the test ran on the CPU at a size that was known to fit.

## 3. First, check the saved predictions

Our model's predictions are recomputed here. They should match the saved ones.

In [2]:
ORDER = ["Elo", "Ranking", "Serve and return", "Form", "Fatigue and fitness", "Experience",
         "Style and age", "Head to head", "Draw", "Home"]
COLUMNS = ([c for group in ORDER for c in FEATURE_GROUPS[group]]
           + list(INTERACTIONS) + DOMAIN_GROUPS["Tournament history"])
YEARS = [2019, 2020, 2021]

conn = sqlite3.connect(Path("../data/processed/tennis.db"))
data = pd.read_sql_query("SELECT * FROM final_features WHERE tourney_level <> 'D' AND year <= 2021", conn)
data = add_interactions(data)

current = data[data["year"].isin(YEARS)].reset_index(drop=True)
assert (current["tourney_id"] == saved["tourney_id"]).all() and (current["match_num"] == saved["match_num"]).all()
ours = walk_forward(data, COLUMNS, logistic_model, years=YEARS).dropna().to_numpy()
print(f"Columns: {len(COLUMNS)}. Largest difference from the saved predictions: {np.abs(ours - saved['ours']).max():.6f}")

Columns: 60. Largest difference from the saved predictions: 0.000000


## 4. The result

Log loss can be split into one value per match, so the two models can be compared match by match. **The gain** is our loss minus Kumo's, which makes it positive when Kumo is better. Resampling matches 10,000 times gives the interval.

In [3]:
gain = np.log(saved["kumo"]) - np.log(saved["ours"])
rng = np.random.default_rng(0)
draws = [gain.to_numpy()[rng.integers(0, len(gain), len(gain))].mean() for _ in range(10000)]
low, high = np.percentile(draws, [2.5, 97.5])

display(pd.DataFrame([{"model": "Our model", **score(saved["ours"])}, {"model": "Kumo", **score(saved["kumo"])}]))
print(f"Gain for Kumo: {gain.mean():+.5f} (95% interval: {low:+.5f} to {high:+.5f}). Passes the rule: {low > 0}")

,model,accuracy_pct,log_loss
0,Our model,67.29,0.5942
1,Kumo,67.38,0.5951


Gain for Kumo: -0.00083 (95% interval: -0.00342 to +0.00186). Passes the rule: False


In [4]:
rows = []
for year, part in saved.groupby("year"):
    rows.append({"year": year, "matches": len(part), "our_log_loss": score(part["ours"])["log_loss"],
                 "kumo_log_loss": score(part["kumo"])["log_loss"], "our_accuracy": score(part["ours"])["accuracy_pct"],
                 "kumo_accuracy": score(part["kumo"])["accuracy_pct"]})
pd.DataFrame(rows)

,year,matches,our_log_loss,kumo_log_loss,our_accuracy,kumo_accuracy
0,2019,2529,0.6060,0.6026,66.43,67.66
1,2020,1347,0.5813,0.5860,67.48,67.19
2,2021,2584,0.5894,0.5924,68.03,67.22


**Finding:** Kumo fails the rule. The interval includes zero, so the data can't separate the two models. Kumo is better in 2019 and worse in 2020 and 2021, which also doesn't suggest a stable advantage.

## 5. Exploring the difference

Nothing in this section is part of the rule.

In [5]:
def logit(p):
    return np.log(p / (1 - p))


def calibration_slope(p):
    """How much the log-odds should be stretched. 1 is ideal, and below 1 means too confident."""
    slopes = np.linspace(0.3, 1.5, 241)
    return slopes[np.argmax([-np.log1p(np.exp(-s * logit(p))).mean() for s in slopes])]


print(f"Same favourite: {100 * ((saved['ours'] > 0.5) == (saved['kumo'] > 0.5)).mean():.1f}% of matches")
print(f"Correlation of log-odds: {np.corrcoef(logit(saved['ours']), logit(saved['kumo']))[0, 1]:.3f}")
print(f"Calibration slope: our model {calibration_slope(saved['ours']):.3f}, Kumo {calibration_slope(saved['kumo']):.3f}")
print("Simple average of both:", score((saved["ours"] + saved["kumo"]) / 2))

Same favourite: 92.9% of matches
Correlation of log-odds: 0.969
Calibration slope: our model 0.960, Kumo 1.060
Simple average of both: {'accuracy_pct': 67.55, 'log_loss': 0.5932}


**Finding:**
- **The two models say nearly the same thing.** This suggests that the limit may be the information we give them, not the model. It doesn't prove it, since a model with other data could behave differently.
- **Kumo's probabilities look well calibrated with 8 estimators.** Its larger gap on 2018 came from 1 estimator, which may have overstated the difference, though 2018 is also a different year.
- **The average of both scores slightly better.** But this idea came after seeing the results, so it would need its own test.

## 6. Conclusions

| Question | Answer |
|---|---|
| Is Kumo better than our model? | Not shown: the difference is close to zero, and its interval includes zero |
| Does it need our feature work? | Not tested: it received the same 60 columns, built in the earlier notebooks |
| Does it beat Pinnacle? | Not tested: the rule only allowed this if Kumo passed |
| Should we replace our model? | No: it's slower and harder to explain, with no measured gain |

**Limits**
- **Our model has a home advantage.** Its columns were chosen with 2015–2021 as validation years, so these years aren't new to it. They are new to Kumo.
- **Kumo saw few matches.** A larger GPU could use many more, and that wasn't tested here.
- **Kumo has some randomness.** Running the same 100 matches twice changed the probabilities by about 0.5 points on average, and no random seed was fixed. A rerun could shift these results a little.
- **One setting was tested.** This result applies to this setting, not to Kumo in general.